## Colab data mounting

1. Upload or copy the extracted `UCI-HAR Dataset` folder into Google Drive.
2. A typical location is `MyDrive/paiml_lab2/UCI-HAR Dataset`.
3. In Colab, run the next cell to mount Drive.
4. If your folder lives somewhere else, update `DRIVE_DATA_ROOT` in the config cell.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Not running inside Colab; skipping Google Drive mount.')

In [ ]:
from pathlib import Path
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/paiml_lab2/UCI-HAR Dataset')
LOCAL_DATA_ROOT = Path('UCI-HAR Dataset')
DATA_ROOT = DRIVE_DATA_ROOT if DRIVE_DATA_ROOT.exists() else LOCAL_DATA_ROOT
assert DATA_ROOT.exists(), f'Dataset folder not found: {DATA_ROOT}'

ARTIFACT_DIR = Path('/content/drive/MyDrive/paiml_lab2_artifacts') if IN_COLAB else Path('artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACT_DIR / 'best_gru_uci_har.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DATA_ROOT =', DATA_ROOT)
print('ARTIFACT_DIR =', ARTIFACT_DIR)
print('DEVICE =', DEVICE)

In [ ]:
def load_activity_map(root: Path):
    names = [line.strip().split()[1] for line in (root / 'activity_labels.txt').read_text().splitlines() if line.strip()]
    return {idx + 1: name for idx, name in enumerate(names)}

def load_inertial_split(root: Path, split: str):
    split_dir = root / split / 'Inertial Signals'
    signal_files = sorted(split_dir.glob(f'*_{split}.txt'))
    X = np.stack([np.loadtxt(path, dtype=np.float32) for path in signal_files], axis=-1)
    y = np.loadtxt(root / split / f'y_{split}.txt', dtype=np.int64)
    subjects = np.loadtxt(root / split / f'subject_{split}.txt', dtype=np.int64)
    return X, y, subjects, [path.name for path in signal_files]

def make_subject_disjoint_fit_val_split(X, y, subjects, val_subject_fraction=0.2, split_seed=42):
    rng = np.random.default_rng(split_seed)
    unique_subjects = np.unique(subjects)
    shuffled = rng.permutation(unique_subjects)
    n_val_subjects = max(1, int(round(val_subject_fraction * len(unique_subjects))))
    val_subjects = set(shuffled[:n_val_subjects].tolist())
    fit_subjects = set(shuffled[n_val_subjects:].tolist())

    fit_mask = np.isin(subjects, list(fit_subjects))
    val_mask = np.isin(subjects, list(val_subjects))
    return {
        'X_fit': X[fit_mask],
        'y_fit': y[fit_mask],
        'subjects_fit': subjects[fit_mask],
        'X_val': X[val_mask],
        'y_val': y[val_mask],
        'subjects_val': subjects[val_mask],
        'fit_subjects': sorted(fit_subjects),
        'val_subjects': sorted(val_subjects),
    }

def compute_channel_stats(X):
    mean = X.mean(axis=(0, 1), keepdims=True)
    std = X.std(axis=(0, 1), keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)

class WindowDataset(Dataset):
    def __init__(self, X, y, mean, std):
        self.X = ((X - mean) / std).astype(np.float32)
        self.y = (y - 1).astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)

activity_map = load_activity_map(DATA_ROOT)
class_names = [activity_map[i] for i in sorted(activity_map)]
X_train, y_train, subj_train, signal_names = load_inertial_split(DATA_ROOT, 'train')
X_test, y_test, subj_test, _ = load_inertial_split(DATA_ROOT, 'test')
split = make_subject_disjoint_fit_val_split(X_train, y_train, subj_train, val_subject_fraction=0.2, split_seed=SEED)
mean_fit, std_fit = compute_channel_stats(split['X_fit'])

print('Signals:', signal_names)
print('Train windows:', X_train.shape, 'subjects:', len(np.unique(subj_train)))
print('Fit windows:', split['X_fit'].shape, 'subjects:', len(split['fit_subjects']))
print('Val windows:', split['X_val'].shape, 'subjects:', len(split['val_subjects']))
print('Test windows:', X_test.shape, 'subjects:', len(np.unique(subj_test)))
print('Class names:', class_names)

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2 if IN_COLAB else 0

fit_ds = WindowDataset(split['X_fit'], split['y_fit'], mean_fit, std_fit)
val_ds = WindowDataset(split['X_val'], split['y_val'], mean_fit, std_fit)
test_ds = WindowDataset(X_test, y_test, mean_fit, std_fit)

fit_loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

class HARSequenceModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes, dropout=0.3, rnn_type='gru'):
        super().__init__()
        rnn_cls = nn.GRU if rnn_type.lower() == 'gru' else nn.LSTM
        self.rnn_type = rnn_type.lower()
        self.rnn = rnn_cls(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        output, hidden = self.rnn(x)
        if self.rnn_type == 'lstm':
            hidden = hidden[0]
        last_hidden = hidden[-1]
        return self.head(self.dropout(last_hidden))

model = HARSequenceModel(
    input_dim=X_train.shape[-1],
    hidden_dim=64,
    num_layers=2,
    num_classes=len(class_names),
    dropout=0.3,
    rnn_type='gru',
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(model)
print('Regularization: dropout=0.3, weight_decay=1e-4')
print('Training robustness: gradient clipping max_norm=1.0, ReduceLROnPlateau scheduler, early stopping')

In [ ]:
def run_epoch(model, loader, optimizer=None, grad_clip=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_targets = []
    all_preds = []

    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        all_targets.append(yb.detach().cpu().numpy())
        all_preds.append(logits.argmax(dim=1).detach().cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return avg_loss, acc, y_true, y_pred

MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 6
GRAD_CLIP = 1.0

history = []
best_state = None
best_val_loss = float('inf')
best_epoch = -1
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc, _, _ = run_epoch(model, fit_loader, optimizer=optimizer, grad_clip=GRAD_CLIP)
    val_loss, val_acc, _, _ = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'lr': lr,
    })
    print(f"epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f} lr={lr:.6f}")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save({
            'model_state_dict': best_state,
            'mean': mean_fit,
            'std': std_fit,
            'class_names': class_names,
            'signal_names': signal_names,
            'config': {
                'input_dim': X_train.shape[-1],
                'hidden_dim': 64,
                'num_layers': 2,
                'num_classes': len(class_names),
                'dropout': 0.3,
                'rnn_type': 'gru',
            },
        }, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('Early stopping triggered.')
            break

assert best_state is not None
model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
print('Best epoch:', best_epoch)
print('Best validation loss:', round(best_val_loss, 4))
print('Checkpoint saved to:', CHECKPOINT_PATH)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_acc'], label='train')
axes[1].plot(history_df['epoch'], history_df['val_acc'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
def predict_from_loader(model, loader):
    model.eval()
    probs_list = []
    pred_list = []
    target_list = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)
            probs_list.append(probs.cpu().numpy())
            pred_list.append(probs.argmax(dim=1).cpu().numpy())
            target_list.append(yb.numpy())
    return np.concatenate(target_list), np.concatenate(pred_list), np.concatenate(probs_list)

y_test_true_idx, y_test_pred_idx, y_test_probs = predict_from_loader(model, test_loader)
test_accuracy = accuracy_score(y_test_true_idx, y_test_pred_idx)
cm = confusion_matrix(y_test_true_idx, y_test_pred_idx)
report = classification_report(y_test_true_idx, y_test_pred_idx, target_names=class_names, output_dict=True, digits=4)
per_class_df = pd.DataFrame(report).T.iloc[:len(class_names)][['precision', 'recall', 'f1-score', 'support']]

print(f'Test accuracy on held-out test subjects: {test_accuracy:.4f}')
display(per_class_df)
print('Confusion matrix (rows=true, cols=pred):')
print(cm)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Test Confusion Matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Error Analysis
- The GRU performed well overall. Perfect performance on Layer class with an accuracy of 1.0 and f1 score of 1.0. The model had great performance will all the walking classes. There was ambiguity in the sitting and standing classes probably because they are the most similar to each other. No directional movemenet but not as positionaly different as laying. 

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
cpu_model = HARSequenceModel(**checkpoint['config'])
cpu_model.load_state_dict(checkpoint['model_state_dict'])
cpu_model.eval()
cpu_mean = checkpoint['mean']
cpu_std = checkpoint['std']

def predict_activity(window):
    window = np.asarray(window, dtype=np.float32)
    expected_shape = (128, checkpoint['config']['input_dim'])
    assert window.shape == expected_shape, f'Expected window shape {expected_shape}, got {window.shape}'
    x = ((window[None, ...] - cpu_mean) / cpu_std).astype(np.float32)
    with torch.no_grad():
        logits = cpu_model(torch.from_numpy(x))
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    label = class_names[int(np.argmax(probs))]
    return label, probs

sample_label, sample_probs = predict_activity(X_test[0])
print('Sample prediction label:', sample_label)
print('Sample probabilities:', np.round(sample_probs, 4))

torch.set_num_threads(1)
num_windows = min(200, len(X_test))
latency_windows = X_test[:num_windows]
start = time.perf_counter()
for window in latency_windows:
    predict_activity(window)
elapsed = time.perf_counter() - start
avg_latency_ms = 1000.0 * elapsed / num_windows
throughput = num_windows / elapsed

print(f'Average CPU inference latency per window over {num_windows} windows: {avg_latency_ms:.3f} ms')
print(f'Approximate throughput: {throughput:.1f} windows/sec')
if avg_latency_ms < 100:
    print('Interpretation: this is feasible for near-real-time use because a 128-step UCI HAR window spans 2.56 seconds, so sub-100 ms model latency leaves substantial headroom.')
else:
    print('Interpretation: this may still be usable offline or batched, but near-real-time deployment would benefit from a smaller hidden size, quantization, or compiled inference.')

### Limitations
- Intended Use: educational human activity recognition. Model training benchmarking. 
- Non-intended use: health specific applications. high stakes: fall detection, emergency response. benchmarking and experimentation only. 
- Evaluation Protocol: using the UCI HAR test and validation subset. 
- Known failures: Data for sitting and standing seems to be similar as shown by the confusion matrix. bad recall. 
- Privacy notice: tracking human activity can always lead to documentation of personal habits. there is no inherently personal identification labeled with the dataset so it's probably all good. 